# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [7]:
# Goal: Define the baseline rule in plain language and document the reason codes the score can emit.

baseline_rule = (
    "Score pages for refresh priority by combining visibility, freshness risk, position opportunity, and content depth."
)

reason_code_definitions = {
    "stale_visible_page": "Visible pages with high impressions and a long time since last update.",
    "declining_with_demand": "Pages losing traffic while still receiving enough demand to justify review.",
    "thin_visible_page": "Low word count pages with enough impressions to benefit from content expansion.",
    "page_one_decay_risk": "High-ranking pages that are old enough to risk losing position without refresh.",
    "low_ctr_visible_page": "Visible pages with enough impressions but below-average click-through rate.",
    "low_engagement_visible_page": "Pages with sessions but weak engagement or scrolling behavior.",
    "general_refresh_review": "Fallback review reason when no stronger signal applies.",
}

baseline_rule, reason_code_definitions

('Score pages for refresh priority by combining visibility, freshness risk, position opportunity, and content depth.',
 {'stale_visible_page': 'Visible pages with high impressions and a long time since last update.',
  'declining_with_demand': 'Pages losing traffic while still receiving enough demand to justify review.',
  'thin_visible_page': 'Low word count pages with enough impressions to benefit from content expansion.',
  'page_one_decay_risk': 'High-ranking pages that are old enough to risk losing position without refresh.',
  'low_ctr_visible_page': 'Visible pages with enough impressions but below-average click-through rate.',
  'low_engagement_visible_page': 'Pages with sessions but weak engagement or scrolling behavior.',
  'general_refresh_review': 'Fallback review reason when no stronger signal applies.'})

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [8]:
# Goal: Build the baseline ranked queue, write the output CSV, and keep the scoring transparent.

from pathlib import Path
import sys

import numpy as np
import pandas as pd

ROOT = Path().resolve()
while not (ROOT / "scripts").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Could not find repository root containing the scripts directory.")
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

from scripts import ml_utils

INPUT_PATH = ROOT / "data" / "processed" / "refresh_feature_vector.csv"
OUTPUT_PATH = ROOT / "work" / "outputs" / "baseline_action_score.csv"
RAW_PATH = ROOT / "data" / "raw" / "content_refresh_anonymized.csv"


def prepare_feature_vector(input_path: Path, output_path: Path) -> pd.DataFrame:
    df = pd.read_csv(input_path)

    numeric_columns = [
        "search_volume",
        "competition",
        "cpc",
        "word_count",
        "char_count",
        "impressions_90d",
        "clicks_90d",
        "pageviews_90d",
        "sessions_90d",
        "users_90d",
        "engaged_sessions_90d",
        "ai_sessions_90d",
        "scroll_events_90d",
        "days_with_impressions",
        "days_with_sessions",
        "impressions_last_30d",
        "clicks_last_30d",
        "sessions_last_30d",
        "impressions_prev_30d",
        "clicks_prev_30d",
        "sessions_prev_30d",
        "content_age_days",
        "age_tier_order",
        "days_since_last_update",
        "ctr",
        "avg_position",
        "engagement_rate",
        "scroll_rate",
        "ai_traffic_pct",
        "trend_pct",
    ]

    categorical_columns = [
        "competition_level",
        "content_type",
        "main_intent",
        "provider_used",
        "model_used",
        "age_tier",
        "freshness_tier",
        "word_count_tier",
        "char_count_tier",
        "impression_tier",
        "position_tier",
        "trend_direction",
    ]

    for column in numeric_columns:
        if column in df.columns:
            df[column] = pd.to_numeric(df[column], errors="coerce")
        else:
            df[column] = 0

    for column in categorical_columns:
        if column in df.columns:
            df[column] = df[column].fillna("unknown").astype(str).replace({"": "unknown", "nan": "unknown"})
        else:
            df[column] = "unknown"

    for column in numeric_columns:
        df[column] = df[column].replace([np.inf, -np.inf], np.nan).fillna(0)

    df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
    df = df.drop_duplicates(subset=["content_id"]).reset_index(drop=True)
    df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

    df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
    df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
    df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
    df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])
    df["has_clicks"] = (df["clicks_90d"] > 0).astype(int)
    df["has_ai_sessions"] = (df["ai_sessions_90d"] > 0).astype(int)
    df["measurable_opportunity"] = (
        (df["impressions_90d"] >= 100) & (df["sessions_90d"] > 0)
    ).astype(int)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_path, index=False)
    return df


if not INPUT_PATH.exists():
    if not RAW_PATH.exists():
        raise FileNotFoundError(f"Missing raw data at {RAW_PATH}")
    df = prepare_feature_vector(RAW_PATH, INPUT_PATH)
else:
    df = pd.read_csv(INPUT_PATH)

if df.empty:
    raise ValueError("Input feature vector is empty.")

# Score components.
df["visibility_score"] = ml_utils.percentile_rank(np.log1p(df["impressions_90d"]))
df["freshness_risk_score"] = ml_utils.percentile_rank(df["days_since_last_update"])
df["position_opportunity_score"] = (
    (1 - ml_utils.normalize(df["avg_position"].clip(lower=1, upper=50)))
    * df["visibility_score"]
    * (df["avg_position"] > 0).astype(int)
)
df["depth_gap_score"] = (1 - ml_utils.percentile_rank(df["word_count"])) * df["visibility_score"]

# Final baseline score and ranking.
df["baseline_refresh_score"] = (
    0.40 * df["visibility_score"]
    + 0.30 * df["freshness_risk_score"]
    + 0.25 * df["position_opportunity_score"]
    + 0.05 * df["depth_gap_score"]
).clip(0, 1)

df["reason_codes"] = df.apply(lambda row: "|".join(reason_codes(row)), axis=1)
df["suggested_action_baseline"] = df.apply(suggested_action, axis=1)
df["baseline_rank"] = df["baseline_refresh_score"].rank(method="first", ascending=False).astype(int)

output_columns = [
    "content_id",
    "client_id",
    "baseline_rank",
    "baseline_refresh_score",
    "visibility_score",
    "freshness_risk_score",
    "position_opportunity_score",
    "depth_gap_score",
    "reason_codes",
    "suggested_action_baseline",
    "is_declining_label",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "avg_position",
    "ctr",
    "engagement_rate",
    "scroll_rate",
    "content_age_days",
    "days_since_last_update",
    "word_count",
    "trend_direction",
]

baseline_queue = df[output_columns].sort_values("baseline_rank")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
baseline_queue.to_csv(OUTPUT_PATH, index=False)

baseline_queue.head(5)

,content_id,client_id,baseline_rank,baseline_refresh_score,visibility_score,freshness_risk_score,position_opportunity_score,depth_gap_score,reason_codes,suggested_action_baseline,...,clicks_90d,sessions_90d,avg_position,ctr,engagement_rate,scroll_rate,content_age_days,days_since_last_update,word_count,trend_direction
21565,content_9532f197bbc8,client_4e07408562,1,0.941189,0.999633,0.8432,0.979233,0.871347,declining_with_demand|page_one_decay_risk|low_...,refresh,...,2689,1098,2.0,0.87,8.01,28.75,445,104,0.0,down
4644,content_4d1fe5b32dc2,client_19581e27de,2,0.934889,0.994167,0.8432,0.963733,0.866582,page_one_decay_risk|low_engagement_visible_page,monitor,...,512,549,2.5,0.52,7.47,13.15,329,104,0.0,stable
18954,content_07f2e7a6f38a,client_19581e27de,3,0.934080,0.994467,0.8432,0.959965,0.866843,page_one_decay_risk|low_engagement_visible_page,monitor,...,856,780,2.7,0.85,2.05,4.60,313,104,0.0,stable
17400,content_e5ae436f9a16,client_4e07408562,4,0.933606,0.996000,0.8432,0.955347,0.868180,page_one_decay_risk|low_ctr_visible_page|low_e...,refresh_and_review_ctr,...,533,522,3.0,0.45,7.09,12.60,421,104,0.0,stable
9348,content_3430a8b94511,client_19581e27de,5,0.933559,0.998167,0.8432,0.951314,0.870069,page_one_decay_risk|low_ctr_visible_page|low_e...,refresh_and_review_ctr,...,440,534,3.3,0.29,6.18,11.04,329,104,0.0,stable


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [9]:
# Goal: Review the top 20 baseline recommendations with confidence and error-mode commentary.

def confidence_note(row: pd.Series) -> str:
    if row["baseline_refresh_score"] >= 0.80:
        return "High confidence: strong visibility and freshness signals."
    if row["baseline_refresh_score"] >= 0.60:
        return "Medium confidence: rule fits but score is not extreme."
    return "Lower confidence: relies on weaker signals or fallback review reason."


def what_would_make_it_wrong(row: pd.Series) -> str:
    if row["suggested_action_baseline"] == "monitor":
        return "May be wrong if demand increases or the page is more important than current metrics show."
    if "declining_with_demand" in row["reason_codes"]:
        return "Could be wrong if the traffic decline is seasonal or caused by temporary ranking noise."
    if "thin_visible_page" in row["reason_codes"]:
        return "Could be wrong if the page already has sufficient content and only needs minor cleanup."
    return "Could be wrong if the engagement or position signals are noisy."


top_20_review = baseline_queue.head(20).copy()
top_20_review["confidence_note"] = top_20_review.apply(confidence_note, axis=1)
top_20_review["what_would_make_it_wrong"] = top_20_review.apply(what_would_make_it_wrong, axis=1)

display_columns = [
    "baseline_rank",
    "content_id",
    "suggested_action_baseline",
    "reason_codes",
    "baseline_refresh_score",
    "confidence_note",
    "what_would_make_it_wrong",
]
top_20_review[display_columns]

,baseline_rank,content_id,suggested_action_baseline,reason_codes,baseline_refresh_score,confidence_note,what_would_make_it_wrong
21565,1,content_9532f197bbc8,refresh,declining_with_demand|page_one_decay_risk|low_...,0.941189,High confidence: strong visibility and freshne...,Could be wrong if the traffic decline is seaso...
4644,2,content_4d1fe5b32dc2,monitor,page_one_decay_risk|low_engagement_visible_page,0.934889,High confidence: strong visibility and freshne...,May be wrong if demand increases or the page i...
18954,3,content_07f2e7a6f38a,monitor,page_one_decay_risk|low_engagement_visible_page,0.934080,High confidence: strong visibility and freshne...,May be wrong if demand increases or the page i...
17400,4,content_e5ae436f9a16,refresh_and_review_ctr,page_one_decay_risk|low_ctr_visible_page|low_e...,0.933606,High confidence: strong visibility and freshne...,Could be wrong if the engagement or position s...
9348,5,content_3430a8b94511,refresh_and_review_ctr,page_one_decay_risk|low_ctr_visible_page|low_e...,0.933559,High confidence: strong visibility and freshne...,Could be wrong if the engagement or position s...
25409,6,content_cbd93118300b,refresh_and_review_ctr,declining_with_demand|page_one_decay_risk|low_...,0.933263,High confidence: strong visibility and freshne...,Could be wrong if the traffic decline is seaso...
18458,7,content_9c195417f6ef,monitor,page_one_decay_risk|low_engagement_visible_page,0.932991,High confidence: strong visibility and freshne...,May be wrong if demand increases or the page i...
13306,8,content_ba2acb4ebd04,monitor,page_one_decay_risk|low_engagement_visible_page,0.931623,High confidence: strong visibility and freshne...,May be wrong if demand increases or the page i...
28354,9,content_79b25654070a,refresh_and_review_ctr,page_one_decay_risk|low_ctr_visible_page|low_e...,0.931363,High confidence: strong visibility and freshne...,Could be wrong if the engagement or position s...
8275,10,content_adddad39251c,monitor,page_one_decay_risk|low_engagement_visible_page,0.931124,High confidence: strong visibility and freshne...,May be wrong if demand increases or the page i...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [10]:
# Goal: Surface weaker baseline choices and confirm the ranking logic did not use future-window or product-flag leakage.

product_flags = [col for col in df.columns if "provider" in col or "model_used" in col or "product" in col]
future_window_columns = [
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "trend_pct",
]
leakage_columns = [col for col in future_window_columns + product_flags if col in df.columns]

weak_picks = baseline_queue[
    (baseline_queue["baseline_refresh_score"] < 0.35)
    | (baseline_queue["suggested_action_baseline"] == "monitor")
].head(20)

weak_picks_summary = weak_picks[
    [
        "baseline_rank",
        "content_id",
        "baseline_refresh_score",
        "suggested_action_baseline",
        "reason_codes",
    ]
]

leakage_report = pd.DataFrame(
    [
        {
            "check": "future-window leak",
            "result": "found" if any(col in future_window_columns for col in leakage_columns) else "none found",
        },
        {
            "check": "product flag leak",
            "result": "found" if any(col in product_flags for col in leakage_columns) else "none found",
        },
        {
            "check": "trend label leak",
            "result": "trend_direction is included only for review output, not as a scoring input",
        },
    ]
)

weak_picks_summary, leakage_report

(       baseline_rank            content_id  baseline_refresh_score  \
 4644               2  content_4d1fe5b32dc2                0.934889   
 18954              3  content_07f2e7a6f38a                0.934080   
 18458              7  content_9c195417f6ef                0.932991   
 13306              8  content_ba2acb4ebd04                0.931623   
 8275              10  content_adddad39251c                0.931124   
 16959             17  content_9351f948bf45                0.930058   
 26935             18  content_37106924f264                0.929529   
 4495              19  content_f4c93868660b                0.929302   
 11733             21  content_140e1efff17e                0.928988   
 26921             30  content_d191a803e7b2                0.926662   
 12780             36  content_88447c37aea0                0.924902   
 13523             37  content_cd36fc9f75ce                0.924731   
 2241              46  content_b6d2061fcd11                0.923358   
 11731

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.